In [1]:
import pandas as pd
import numpy as np
from ml_utils.src import *
from ml_utils.config import *

from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando SVM

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df_treino = pd.read_parquet(DATA_DIR, columns = colunas)
df_teste = pd.read_parquet("2023", columns = colunas)

## 1.2- Pré-Processando os Dados

In [3]:
df_treino = preparar_dados(df_treino, objetivo = 'Desempenho', n_samples = 40000)
df_teste = preparar_dados(df_teste, objetivo = 'Desempenho', n_samples = 10000)

## 1.3- Construção da Matriz X e Vetor y

In [4]:
X_treino = df_treino.drop(columns=['MEDIA', 'FALTOU'])
y_treino = df_treino['MEDIA']

X_teste = df_teste.drop(columns=['MEDIA', 'FALTOU'])
y_teste = df_teste['MEDIA']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_treino, y_treino, test_size=0.2, random_state=42)

In [6]:
quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_val   = (y_val   >= quantil).astype(int)

y_train = y_train.map({0: -1, 1: 1})
y_val = y_val.map({0: -1, 1: 1})


## 1.5 - Tratando os Dados

In [7]:
preprocessador = pre_processor(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)

## 1.6 - Treinando o SVM com os melhores parâmetros

In [9]:
pipe = Pipeline([
    ('svm', SVC(kernel='rbf', random_state=42))
])

params = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(pipe, params, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)  

print(grid.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'svm__C': 1, 'svm__gamma': 0.1}


## 1.7 - Validando o SVM

In [10]:
print(classification_report(y_val, grid.predict(X_val)))

              precision    recall  f1-score   support

          -1       0.70      0.73      0.72       853
           1       0.71      0.67      0.69       829

    accuracy                           0.70      1682
   macro avg       0.70      0.70      0.70      1682
weighted avg       0.70      0.70      0.70      1682



## 1.8 Treinando o SVM com Todos os Dados

In [11]:
quantil = y_treino.quantile(0.5)
y_treino = (y_treino >= quantil).astype(int)
y_teste  = (y_teste  >= quantil).astype(int)

y_treino = y_treino.map({0: -1, 1: 1})
y_teste = y_teste.map({0: -1, 1: 1})

X_treino = preprocessador.fit_transform(X_treino).astype(np.float32)
X_teste  = preprocessador.transform(X_teste).astype(np.float32) 

svm = SVC(kernel='rbf', random_state=42, C = 1, gamma= 0.1)

svm.fit(X_treino, y_treino)

print(classification_report(y_teste, svm.predict(X_teste)))

              precision    recall  f1-score   support

          -1       0.63      0.77      0.69      1148
           1       0.76      0.62      0.69      1368

    accuracy                           0.69      2516
   macro avg       0.70      0.70      0.69      2516
weighted avg       0.70      0.69      0.69      2516

